# 15. SQL Views & Materialized Caching: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **15. SQL Views & Materialized Caching**. Database views encapsulate complex business logic behind reusable, simplified virtual interfaces. This notebook covers standard virtual views (`CREATE VIEW`), simulated materialized views, caching refresh policies, and query optimizer view folding mechanics.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Virtual View Abstraction: `CREATE VIEW view_name AS ...`
- [x] 🔹 Materialized Result Caching: Physical Storage & Refresh Policies
- [x] 🔹 Query Optimizer View Expansion & Predicate Pushdown
- [x] 🔍 Scenario: Executive Financial Risk Reporting Abstraction Layer








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Virtual View Abstraction: `CREATE VIEW`
- **What it does:** Saves a declarative query definition in the data catalog as a reusable virtual table without duplicating physical data.
- **Syntax:** `CREATE VIEW view_name AS SELECT ... FROM ...`
- **Dataset Application & Code Demonstration:** Encapsulates customer transaction summaries in an executive view.


In [2]:
%%sql
CREATE VIEW IF NOT EXISTS v_customer_financial_summary AS
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier,
    COUNT(t.transaction_id) AS total_transactions,
    ROUND(COALESCE(SUM(t.transaction_amount), 0), 2) AS total_spent,
    SUM(CASE WHEN t.is_fraud = 1 THEN 1 ELSE 0 END) AS fraud_event_count
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.account_tier;

SELECT * FROM v_customer_financial_summary WHERE total_transactions > 0 LIMIT 5;


'Query Executed Successfully.'

### 🔹 Materialized Result Caching Concepts
- **What it does:** Persists the computed result set of a complex analytical view physically to disk, eliminating expensive runtime join recalculations.
- **Syntax:** `CREATE MATERIALIZED VIEW mat_view_name AS ...`
- **Dataset Application & Code Demonstration:** Simulates a materialized summary cache using a persisted summary table.


In [3]:
%%sql
CREATE TABLE IF NOT EXISTS mat_regional_metrics_cache AS
SELECT 
    region,
    COUNT(transaction_id) AS total_volume_count,
    ROUND(SUM(transaction_amount), 2) AS gross_volume,
    ROUND(AVG(transaction_amount), 2) AS avg_ticket_size,
    CURRENT_TIMESTAMP AS last_refreshed_at
FROM transactions
WHERE region IS NOT NULL
GROUP BY region;

SELECT * FROM mat_regional_metrics_cache;


'Query Executed Successfully.'

## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Virtual View vs Materialized View Architectural Tradeoffs
- **Objective:** Evaluate storage footprint, write latency, read performance, and cache staleness tradeoffs between Virtual Views and Materialized Views.
- **Approach:** Generate a decision matrix comparing both architectural patterns.


In [4]:
%%sql
SELECT 
    'Virtual View (CREATE VIEW)' AS view_type, 'Zero (Query definition only)' AS storage_cost, 'Always Fresh (Real-time)' AS data_freshness, 'Recomputes query on each read' AS query_latency
UNION ALL
SELECT 
    'Materialized View (Physical)', 'Stores full result table on disk', 'Requires REFRESH schedule', 'Blazing Fast (Pre-computed disk read)';


,view_type,storage_cost,data_freshness,query_latency
0,Virtual View (CREATE VIEW),Zero (Query definition only),Always Fresh (Real-time),Recomputes query on each read
1,Materialized View (Physical),Stores full result table on disk,Requires REFRESH schedule,Blazing Fast (Pre-computed disk read)
